In [21]:
import os
import json
import time
import pathlib
import re
from string import Template
from dotenv import load_dotenv
from openai import OpenAI
from pymed import PubMed
from typing import List, Dict


In [22]:
BASE_DIR = pathlib.Path.cwd()

load_dotenv(BASE_DIR / "environment.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY not set in .env")



In [23]:
with open(BASE_DIR / "config.json", "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

PUBMED_CFG = CONFIG["pubmed"]
OPENAI_CFG = CONFIG["openai"]
OUTPUT_CFG = CONFIG["output"]

# Optional manual PubMed IDs and claim override to bypass search
MANUAL_PUBMED_IDS = PUBMED_CFG.get("manual_pubmed_ids", [])
MANUAL_CLAIM = CONFIG.get("manual_claim", {})


In [24]:
OUTPUT_DIR = BASE_DIR / OUTPUT_CFG.get("output_dir")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [25]:
with open(BASE_DIR / "claims.json", "r", encoding="utf-8") as f:
    CLAIMS = json.load(f)

In [26]:
with open(BASE_DIR / "toulmin_prompt.txt", "r", encoding="utf-8") as f:
    TOULMIN_TEMPLATE = Template(f.read())

In [27]:
client = OpenAI(api_key=OPENAI_API_KEY)

pubmed = PubMed(
    tool=PUBMED_CFG["tool"],
    email=PUBMED_CFG["email"]
)

In [28]:
def build_rct_query(drug: str, condition: str, outcome: str) -> str:
    parts = [f'"{drug}"[Title/Abstract]']

    if condition:
        parts.append(f'"{condition}"[Title/Abstract]')
    if outcome:
        parts.append(f'"{outcome}"[Title/Abstract]')

    parts.append(
        '('
        '"randomized controlled trial"[Publication Type] OR '
        '"randomized controlled trial, phase ii"[Publication Type] OR '
        '"randomized controlled trial, phase iii"[Publication Type] OR '
        'randomized[Title/Abstract] OR '
        'placebo[Title/Abstract]'
        ')'
    )

    return " AND ".join(parts)

In [29]:
def build_review_query(drug: str, condition: str, outcome: str) -> str:
    parts = [f'"{drug}"[Title/Abstract]']

    if condition:
        parts.append(f'"{condition}"[Title/Abstract]')
    if outcome:
        parts.append(f'"{outcome}"[Title/Abstract]')

    parts.append(
        '('
        '"systematic review"[Publication Type] OR '
        '"meta-analysis"[Publication Type] OR '
        '"review"[Publication Type] OR '
        '"systematic review"[Title/Abstract] OR '
        '"meta-analysis"[Title/Abstract]'
        ')'
    )

    return " AND ".join(parts)

In [30]:
def normalize_year(year_value) -> int:
    """Coerce a variety of 'year' representations to an int year; return 0 if unknown."""
    if isinstance(year_value, int):
        return year_value
    if isinstance(year_value, str):
        try:
            return int(year_value)
        except Exception:
            m = re.search(r'\d{4}', year_value)
            if m:
                return int(m.group(0))
            return 0
    # datetime/date-like objects
    try:
        y = getattr(year_value, "year", None)
        if isinstance(y, int):
            return y
    except Exception:
        pass
    return 0


def filter_by_year(articles: List[Dict], year_from: int) -> List[Dict]:
    return [a for a in articles if normalize_year(a.get("year")) >= year_from]


In [31]:
def fetch_articles(query: str, max_results: int) -> List[Dict]:
    results = pubmed.query(query, max_results=max_results)
    articles = []
    sleep_s = PUBMED_CFG.get("sleep_between_requests", 0.34)

    for article in results:
        art = article.toDict()
        pub_date = art.get("publication_date")
        year = None
        if pub_date is not None:
            try:
                # pub_date may be a date-like object or string
                year = getattr(pub_date, "year", pub_date)
            except Exception:
                pass

        # Normalize year to an int (0 if unknown)
        try:
            art["year"] = normalize_year(year)
        except Exception:
            art["year"] = 0

        # Ensure publication_types is a list for downstream checks
        pts = art.get("publication_types")
        if pts is None:
            art["publication_types"] = []
        else:
            art["publication_types"] = pts

        # Abstract normalisation
        if isinstance(art.get("abstract"), list):
            art["abstract"] = " ".join(art["abstract"])

        articles.append(art)
        time.sleep(sleep_s)

    return articles


In [32]:
def fetch_articles_by_ids(pubmed_ids):
    """Fetch articles directly by PubMed ID using the existing fetch_articles helper."""
    articles = []
    ids = pubmed_ids or []
    for pid in ids:
        q = f"{pid}[PMID]"
        fetched = fetch_articles(q, max_results=1)
        if fetched:
            articles.append(fetched[0])
    return articles


In [33]:
def is_rct(art: Dict) -> bool:
    """
    Detect RCTs robustly:
    - prefer publication_types when present
    - fallback to title/abstract/methods keywords
    """
    pts = []
    raw_pts = art.get("publication_types") or []
    for pt in raw_pts:
        try:
            pts.append(str(pt).lower())
        except Exception:
            pass

    pt_signals = (
        "randomized controlled trial",
        "randomised controlled trial",
        "randomized trial",
        "randomised trial",
        "randomized clinical trial",
        "randomised clinical trial",
        "double-blind",
        "double blind",
        "placebo-controlled",
        "placebo controlled"
    )
    if any(any(sig in pt for sig in pt_signals) for pt in pts):
        return True

    title = (art.get("title") or "").lower()
    abstract = (art.get("abstract") or "").lower()
    methods = (art.get("methods") or "").lower()
    combined = " ".join([title, abstract, methods])

    kw_signals = (
        "randomized controlled trial",
        "randomised controlled trial",
        "randomized trial",
        "randomised trial",
        "randomized",
        "randomised",
        "randomly",
        "random allocation",
        "double-blind",
        "double blind",
        "placebo-controlled",
        "placebo controlled",
        "placebo"
    )
    return any(k in combined for k in kw_signals)


def is_review(art: Dict) -> bool:
    """
    Detect systematic reviews/meta-analyses:
    - prefer publication_types
    - fallback to title/abstract keywords
    """
    pts = []
    raw_pts = art.get("publication_types") or []
    for pt in raw_pts:
        try:
            pts.append(str(pt).lower())
        except Exception:
            pass

    pt_signals = (
        "systematic review",
        "meta-analysis",
        "meta analysis",
        "review"
    )
    if any(any(sig in pt for sig in pt_signals) for pt in pts):
        return True

    title = (art.get("title") or "").lower()
    abstract = (art.get("abstract") or "").lower()
    text = " ".join([title, abstract])

    if "protocol" in text:
        return False

    kw_signals = (
        "systematic review",
        "meta-analysis",
        "meta analysis",
        "pooled analysis",
        "evidence synthesis"
    )
    return any(k in text for k in kw_signals)


In [34]:
def get_evidence_bundle(drug: str, condition: str, outcome: str) -> Dict:
    # If manual IDs are provided, bypass search and return those articles as RCTs (type unknown)
    if MANUAL_PUBMED_IDS:
        manual_articles = fetch_articles_by_ids(MANUAL_PUBMED_IDS)
        for art in manual_articles:
            art.setdefault("_evidence_type", "manual")
        return {"rcts": manual_articles, "reviews": []}

    year_from = PUBMED_CFG["year_from"]
    max_results = PUBMED_CFG["max_results"]
    max_rcts = PUBMED_CFG["max_rcts"]
    max_reviews = PUBMED_CFG["max_reviews"]

    q_rct = build_rct_query(drug, condition, outcome)
    #print("rct query is", q_rct)
    q_rev = build_review_query(drug, condition, outcome)
    #print("review query is", q_rev)

    rct_candidates = fetch_articles(q_rct, max_results=max_results)
    #print("rct candidates are", rct_candidates)
    review_candidates = fetch_articles(q_rev, max_results=max_results)

    rct_candidates = filter_by_year(rct_candidates, year_from)
    #print('rct candidates are : ', rct_candidates)
    review_candidates = filter_by_year(review_candidates, year_from)
    rcts = [a for a in rct_candidates if is_rct(a)]
    reviews = [a for a in review_candidates if is_review(a)]

    rcts = sorted(rcts, key=lambda a: normalize_year(a.get("year")), reverse=True)[:max_rcts]
    reviews = sorted(reviews, key=lambda a: normalize_year(a.get("year")), reverse=True)[:max_reviews]
    #print('sorted rcts: ',rcts)
    return {"rcts": rcts, "reviews": reviews}


In [35]:
def build_toulmin_prompt(claim_text: str, article: Dict) -> str:
    return TOULMIN_TEMPLATE.substitute(
        claim_text=claim_text,
        pubmed_id=article.get("pubmed_id", ""),
        title=article.get("title", ""),
        abstract=article.get("abstract", "")
    )


In [36]:
def extract_toulmin_argument(article: Dict, claim_text: str) -> Dict:
    prompt = build_toulmin_prompt(claim_text, article)

    response = client.chat.completions.create(
        model=OPENAI_CFG["model"],
        temperature=OPENAI_CFG.get("temperature", 0),
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": "You are a precise biomedical argument extraction assistant."},
            {"role": "user", "content": prompt}
        ]
    )

    content = response.choices[0].message.content
    try:
        parsed = json.loads(content)
    except json.JSONDecodeError:
        parsed = {
            "stance": "parse_error",
            "toulmin": {
                "claim": "",
                "data": [],
                "warrant": "",
                "backing": [],
                "qualifier": "",
                "rebuttals": [],
                "raw": content
            }
        }

    return parsed

In [37]:
def process_claim(claim: Dict):
    claim_id = claim["claim_id"]
    claim_text = claim["claim_text"]
    drug = claim["drug"]
    condition = claim.get("condition", "")
    outcome = claim.get("outcome", "")

    print(f"\n=== Processing claim {claim_id} ===")
    print(f"Drug: {drug}, Condition: {condition}, Outcome: {outcome}")
    print(f"Text: {claim_text}")

    bundle = get_evidence_bundle(drug, condition, outcome)

    all_articles: List[Dict] = []
    for art in bundle["rcts"]:
        art["_evidence_type"] = "rct"
        all_articles.append(art)
    for art in bundle["reviews"]:
        art["_evidence_type"] = "review"
        all_articles.append(art)

    print(f"Total articles to process with LLM: {len(all_articles)}")
    
    out_path = OUTPUT_DIR / f"{claim_id}_toulmin.jsonl"
    with out_path.open("w", encoding="utf-8") as f_out:
        for idx, art in enumerate(all_articles, start=1):
            print(f"[{claim_id}] [{idx}/{len(all_articles)}] PubMed ID {art.get('pubmed_id')}")
            toulmin_arg = extract_toulmin_argument(art, claim_text)

            record = {
                "claim_id": claim_id,
                "claim_text": claim_text,
                "drug": drug,
                "condition": condition,
                "outcome": outcome,
                "source": {
                    "pubmed_id": art.get("pubmed_id"),
                    "title": art.get("title"),
                    "year": art.get("year"),
                    "journal": art.get("journal"),
                    "evidence_type": art.get("_evidence_type"),
                    "abstract": art.get("abstract")
                },
                "stance": toulmin_arg.get("stance"),
                "toulmin": toulmin_arg.get("toulmin", {})
            }

            f_out.write(json.dumps(record, ensure_ascii=False) + "\n")
            time.sleep(0.5)

    print(f"Saved Toulmin arguments for {claim_id} to {out_path}")
    


In [38]:
def run_pipeline():
    # If manual claim + manual PubMed IDs are provided, run only that claim and bypass search
    if MANUAL_PUBMED_IDS and MANUAL_CLAIM:
        print("Manual mode: using manual_pubmed_ids and manual_claim from config.json")
        process_claim(MANUAL_CLAIM)
        return

    active_claims = [c for c in CLAIMS if c.get("active", False)]

    if not active_claims:
        print("No active claims found in claims.json")
        return

    for claim in active_claims:
        process_claim(claim)


In [40]:
run_pipeline()

Manual mode: using manual_pubmed_ids and manual_claim from config.json

=== Processing claim semaglutide_manual ===
Drug: semaglutide, Condition: , Outcome: 
Text: Semaglutide induces significant weight loss in adults with overweight or obesity, but long-term benefit-risk balance depends on continued use and remains under investigation.
Total articles to process with LLM: 6
[semaglutide_manual] [1/6] PubMed ID 33567185
[semaglutide_manual] [2/6] PubMed ID 33625476
24222017
12704397
15321793
21593294
27219496
28455679
23288372
32721163
30421856
32090517
29246950
28385659
30122305
32441473
33462358
26418188
28192109
29156185
26106187
26086031
22029981
32213703
33269530
27860132
31600725
7847427
34192450
[semaglutide_manual] [3/6] PubMed ID 33755728
30844811
26106187
27574404
25197563
27219496
26641646
26868660
28919062
27299618
32441473
33269530
32213703
30122305
30026333
24141714
31168921
33462358
26418188
9683204
20647200
24189773
23532991
32870301
29221645
26154880
26132939
23812094
3

## Cell for debugging

In [ ]:
from pprint import pprint

def debug_evidence(drug, condition, outcome, max_show=5):
    print(f"Querying RCTs for: drug={drug}, condition={condition}, outcome={outcome}\n")
    q_rct = build_rct_query(drug, condition, outcome)
    rct_candidates = fetch_articles(q_rct, max_results=PUBMED_CFG.get('max_results', 20))
    print("Fetched RCT candidates:", len(rct_candidates))
    for i, a in enumerate(rct_candidates[:max_show], start=1):
        print("\n--- Candidate", i, "---")
        print("pubmed_id:", a.get("pubmed_id"))
        print("title:", (a.get("title") or "")[:300])
        print("year (raw):", a.get("year"))
        try:
            print("year (norm):", normalize_year(a.get("year")))
        except Exception as e:
            print("normalize_year error:", e)
        print("publication_types:", a.get("publication_types"))
        print("keys:", list(a.keys()))
        abstract = a.get("abstract") or ""
        print("abstract snippet:", abstract[:300])
        pprint({k: a.get(k) for k in ("journal", "authors")})

    year_from = PUBMED_CFG["year_from"]
    filtered = filter_by_year(rct_candidates, year_from)
    print(f"\nAfter filter_by_year (>= {year_from}): {len(filtered)}")
    rcts = [a for a in filtered if is_rct(a)]
    print("is_rct matched:", len(rcts))
    fallback = [
        a for a in filtered
        if ("random" in (a.get("title") or "").lower() or "random" in (a.get("abstract") or "").lower())
    ]
    print("fallback 'random' matches:", len(fallback))
    return rct_candidates, filtered, rcts, fallback

# Example run using the first active claim
active = [c for c in CLAIMS if c.get("active", False)]
if active:
    c = active[0]
    print("Using claim:", c.get("claim_id"))
    debug_evidence(c["drug"], c.get("condition", ""), c.get("outcome", ""))
else:
    print("No active claims loaded; call debug_evidence(drug, condition, outcome) manually")

Using claim: metformin_valid
Querying RCTs for: drug=metformin, condition=type 2 diabetes, outcome=HbA1c

Fetched RCT candidates: 200

--- Candidate 1 ---
pubmed_id: 41282288
39847507
37287751
25285158
35351622
28213644
38891870
36120427
31982487
30503831
34502360
27928958
35712251
33901926
33445738
36743910
24478399
21909417
36674679
34009375
23586463
27141961
14597658
11846609
23749231
36896775
38153649
34151713
35909199
38616305
26138690
30872315
36895784
35498402
35124182
38848290
38945455
37057737
27059430
23032062
33367814
35590121
26031505
28273212
34631276
25532038
30067154
34208360
title: MicroRNAs modulated by DPP-4 inhibitor and bedtime NPH insulin therapy in individuals with type 2 diabetes.
year (raw): 2025
year (norm): 2025
publication_types: []
keys: ['pubmed_id', 'title', 'abstract', 'keywords', 'journal', 'publication_date', 'authors', 'methods', 'conclusions', 'results', 'copyrights', 'doi', 'xml', 'year', 'publication_types']
abstract snippet: MicroRNAs (miRNAs) and 